In [1]:
import torch as t
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import einops
from jaxtyping import Float, Int
from dataclasses import dataclass
from torch.utils.data import DataLoader
from src import transformer, training, evaluation, utils, data
from tqdm import tqdm
import datasets
from transformers import GPT2Tokenizer
import wandb

/Users/paul-philiplouis/work/transformer-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# With simple learnt added positional embedding

In [ ]:
# Configure device
if t.backends.mps.is_available():
    device = t.device('mps')
elif t.cuda.is_available():
    device = t.device("cuda")
else:
    device = t.device('cpu')
print("Device: ", device)

# Load model
cfg = transformer.Config(d_model = 64, n_ctx=64, d_head = 16, d_mlp = 256, n_heads = 4, n_layers = 4, use_rope=False)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model = transformer.Transformer(tokenizer, cfg).to(device)


Device:  mps


In [ ]:
# Loading data from TinyStories dataset
dataset = datasets.load_dataset("roneneldan/TinyStories", streaming=True, split="train")

token_dataset = data.StreamingTokenDataset(dataset, tokenizer, seq_len=64)
dataloader = DataLoader(token_dataset, batch_size=32, num_workers=0)

In [ ]:
optimizer = t.optim.AdamW(model.parameters(), lr = 1e-3)
scheduler = t.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=utils.get_lr)

In [ ]:
epochs = 5

step = 0
for _ in range(epochs):
    training.training_over_batch(model, dataloader, optimizer, scheduler)

Step 0 | Loss = 10.833
['The little girl had obesity Telecom endeavors crusadeWATCHventures fragmentedTour Saving knife ReedÃÂÃÂ Leah statisticsÃÂÃÂ']
Step 10 | Loss = 10.799
Step 20 | Loss = 10.724
Step 30 | Loss = 10.572
Step 40 | Loss = 10.366
Step 50 | Loss = 10.081
['The little girl had...............']
Step 60 | Loss = 9.765
Step 70 | Loss = 9.473
Step 80 | Loss = 9.007
Step 90 | Loss = 8.621
Step 100 | Loss = 8.074
['The little girl had...............']
Step 110 | Loss = 7.593
Step 120 | Loss = 7.481
Step 130 | Loss = 7.053
Step 140 | Loss = 6.685
Step 150 | Loss = 6.195
['The little girl had..\n\n\n\n\n\n\n\n\n\n\n\n\n']
Step 160 | Loss = 6.172
Step 170 | Loss = 6.003
Step 180 | Loss = 5.969
Step 190 | Loss = 5.943
Step 200 | Loss = 5.566
['The little girl had a.\n\n\n\n\n\n\n\n\n\n\n\n\n']
Step 210 | Loss = 5.488
Step 220 | Loss = 5.330
Step 230 | Loss = 5.163


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


Step 240 | Loss = 5.053
Step 250 | Loss = 4.914
['The little girl had a big. They said.\n\n\n\n\n\n\n\n\n']
Step 260 | Loss = 4.962
Step 270 | Loss = 4.557
Step 280 | Loss = 5.076
Step 290 | Loss = 4.944
Step 300 | Loss = 4.743
['The little girl had a time, "I\'m a time, "I\'m a time,']
Step 310 | Loss = 4.512
Step 320 | Loss = 4.305
Step 330 | Loss = 4.744
Step 340 | Loss = 4.347
Step 350 | Loss = 4.283
['The little girl had a time, there was a time, and he was a time, there']
Step 360 | Loss = 4.518
Step 370 | Loss = 4.318
Step 380 | Loss = 4.482
Step 390 | Loss = 4.470
Step 400 | Loss = 4.354
['The little girl had a time, "I can the park. He was a time, but']
Step 410 | Loss = 4.065
Step 420 | Loss = 4.122
Step 430 | Loss = 4.508
Step 440 | Loss = 4.177
Step 450 | Loss = 4.449
['The little girl had a big, but he was so excited.\n\nThe little girl was']
Step 460 | Loss = 3.816
Step 470 | Loss = 4.247
Step 480 | Loss = 4.133
Step 490 | Loss = 4.130
Step 500 | Loss = 3.847
['The little

KeyboardInterrupt: 

# With RoPE

In [2]:
# Configure device
if t.backends.mps.is_available():
    device = t.device('mps')
elif t.cuda.is_available():
    device = t.device("cuda")
else:
    device = t.device('cpu')
print("Device: ", device)

Device:  mps


In [3]:
import importlib
importlib.reload(transformer)

<module 'src.transformer' from '/Users/paul-philiplouis/work/transformer-training/src/transformer.py'>

In [4]:
# Load model
t.manual_seed(42)
cfg = transformer.Config(d_model = 64, n_ctx=64, d_head = 16, d_mlp = 256, n_heads = 4, n_layers = 4, use_rope=True)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model = transformer.Transformer(tokenizer, cfg).to(device)

In [5]:
# Loading data from TinyStories dataset
dataset = datasets.load_dataset("roneneldan/TinyStories", streaming=True, split="train")

token_dataset = data.StreamingTokenDataset(dataset, tokenizer, seq_len=64)
dataloader = DataLoader(token_dataset, batch_size=32, num_workers=0)
optimizer = t.optim.AdamW(model.parameters(), lr = 1e-3)
scheduler = t.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=utils.get_lr)

In [6]:
epochs = 5

step = 0
for _ in range(epochs):
    training.training_over_batch(model, dataloader, optimizer, scheduler)

Step 0 | Loss = 10.815
['The little girl hadalling chef PARclassmeal skip indistinguishable defied relaxedARR Edmonton Dynamiculy relaxedARR']
Step 10 | Loss = 10.785
Step 20 | Loss = 10.710
Step 30 | Loss = 10.570
Step 40 | Loss = 10.372
Step 50 | Loss = 10.087
['The little girl had...............']
Step 60 | Loss = 9.749
Step 70 | Loss = 9.449
Step 80 | Loss = 8.966
Step 90 | Loss = 8.560
Step 100 | Loss = 7.986
['The little girl had...............']
Step 110 | Loss = 7.495
Step 120 | Loss = 7.376
Step 130 | Loss = 6.947
Step 140 | Loss = 6.583
Step 150 | Loss = 6.108
['The little girl had...............']
Step 160 | Loss = 6.109
Step 170 | Loss = 5.956
Step 180 | Loss = 5.912
Step 190 | Loss = 5.896
Step 200 | Loss = 5.521
['The little girl had a.\n\n\n\n\n\n\n\n\n\n\n\n\n']
Step 210 | Loss = 5.446
Step 220 | Loss = 5.299
Step 230 | Loss = 5.139


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


Step 240 | Loss = 5.028
Step 250 | Loss = 4.844
['The little girl had a big.\n\n"I have to the park.\n\n\n']
Step 260 | Loss = 4.887
Step 270 | Loss = 4.464
Step 280 | Loss = 4.978
Step 290 | Loss = 4.821
Step 300 | Loss = 4.613
['The little girl had a time, "I\'m so happy.\n\n\nThe little girl']
Step 310 | Loss = 4.391
Step 320 | Loss = 4.181
Step 330 | Loss = 4.639
Step 340 | Loss = 4.209
Step 350 | Loss = 4.178
['The little girl had a time. He was so happy. He was so happy. He was']
Step 360 | Loss = 4.404
Step 370 | Loss = 4.211
Step 380 | Loss = 4.330
Step 390 | Loss = 4.325
Step 400 | Loss = 4.233
['The little girl had a time.\n\nThe little girl was so happy and said, "']
Step 410 | Loss = 3.971
Step 420 | Loss = 4.024
Step 430 | Loss = 4.409
Step 440 | Loss = 4.078
Step 450 | Loss = 4.393
['The little girl had a big, and he was so excited.\n\nThe little girl was']
Step 460 | Loss = 3.720
Step 470 | Loss = 4.175
Step 480 | Loss = 4.051
Step 490 | Loss = 4.054
Step 500 | Loss = 3.

KeyboardInterrupt: 